# S2 cointegration — Monte Carlo EV vs SPY

Sealed OOS net returns only. **This notebook is EV vs SPY** (expected value, HAC/bootstrap significance of the mean, $P(\mathrm{not\ beat\ SPY})$). It does not compute prop-firm pass rates.


## 0. Imports & Config


In [1]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.monte_carlo.loaders import (
    aligned_strategy_spy,
    find_repo_root,
    load_sealed_s2,
)
from risk.analytics.monte_carlo.report import run_ev_vs_spy

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = 's2'
BAR = 'D'
PERIODS_PER_YEAR = 252
DEFAULT_H = 63
DEFAULT_BLOCK = 10.0
DEFAULT_N_SIM = 400


ROOT c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio


## 1. Data Loading


In [2]:
FRAME = aligned_strategy_spy(load_sealed_s2(ROOT), bar=BAR)
print(FRAME.tail())
print("n_bars", len(FRAME), "start", FRAME.index.min().date(), "end", FRAME.index.max().date())


            strategy       spy
date                          
2026-08-18  0.005848 -0.006756
2026-08-19 -0.003568  0.002098
2026-08-20  0.001238 -0.008400
2026-08-21 -0.001774  0.004091
2026-08-24  0.000000 -0.002938
n_bars 1164 start 2022-01-03 end 2026-08-24


## 2. Historical EV & significance

Headline: **t-stat, p-value, `ci_excludes_zero`**. These describe the *historical mean* of sealed OOS period returns (H0: $E[r]=0$), not a single simulated path. Block-bootstrap $P^*(\hat\mu^*\le 0)$ is the nonparametric counterpart. PSR is secondary Sharpe quality, not the EV-significance metric.


In [3]:
from risk.analytics.monte_carlo.ev_stats import ev_significance, excess_returns
from risk.analytics.monte_carlo.plots import significance_frame

hist = ev_significance(
    FRAME["strategy"],
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=800,
    random_seed=0,
)
excess = ev_significance(
    excess_returns(FRAME["strategy"], FRAME["spy"]),
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=800,
    random_seed=0,
)
print("Strategy EV significance (historical mean)")
display(significance_frame(hist))
print("Excess vs SPY EV significance (distinct from P(not beat SPY))")
display(significance_frame(excess))


Strategy EV significance (historical mean)


,mean,mean_ann,t_stat,p_value,ci_low,ci_high,ci_excludes_zero,bootstrap_p_mean_le_0,bootstrap_ci_low,bootstrap_ci_high,n_obs,psr,psr_vs_1
0,0.000462,0.116498,3.317786,0.000907,0.000189,0.000735,True,0.0,0.000218,0.00072,1164,1.0,0.996904


Excess vs SPY EV significance (distinct from P(not beat SPY))


,mean,mean_ann,t_stat,p_value,ci_low,ci_high,ci_excludes_zero,bootstrap_p_mean_le_0,bootstrap_ci_low,bootstrap_ci_high,n_obs,psr,psr_vs_1
0,-0.000058,-0.014668,-0.175055,0.861037,-0.00071,0.000593,False,0.5925,-0.000688,0.000618,1164,0.005625,0.0


## 3. Joint path simulator

Stationary block bootstrap draws **paired** strategy and SPY paths (same block indices). Independent resampling would invalidate $P(\mathrm{not\ beat\ SPY})$. Leverage $k$ scales **strategy** simple returns only (`r' = k r`); it is a risk-budget overlay, not a re-run of sleeve vol-targeting.


## 4. Interactive horizon / leverage

Gold/red lines are the **sealed OOS** wealth over the last (and, if the sample is longer, first) $H$ bars, drawn on the bootstrap fan. The excess fan is $W_{\mathrm{strat}}/W_{\mathrm{SPY}}$.


In [4]:
PACK = {}

def run(horizon, leverage, n_simulations, mean_block_length, haircut_bps):
    pack = run_ev_vs_spy(
        FRAME,
        n_simulations=int(n_simulations),
        horizon=int(horizon),
        leverage=float(leverage),
        mean_block_length=float(mean_block_length),
        periods_per_year=PERIODS_PER_YEAR,
        random_seed=0,
        n_bootstrap=N_BOOTSTRAP,
        haircut_bps=float(haircut_bps),
    )
    PACK.clear()
    PACK.update(pack)
    print("Headline (no prop-firm pass rates)")
    display(pack["headline"].to_frame("value"))
    print("Historical strategy EV significance")
    display(pack["hist_table"])
    print("Excess vs SPY (expectation), separate from P(not beat SPY)")
    display(pack["excess_table"])
    print("Pathwise holes (max DD / time in hole / recover)")
    display(pack["holes_summary"].to_frame("value"))
    print("EV concentration (mean vs median vs CVaR; top-decile share)")
    display(pack["concentration"].to_frame("value"))
    print("Joint shape vs SPY")
    display(pack["joint_shape"].to_frame("value"))
    display(pack["fan"])
    display(pack["excess_fan"])
    display(pack["max_dd_hist"])
    display(pack["dd_scatter"])
    display(pack["terminals"])
    return pack

N_BOOTSTRAP = 600
pack = run(DEFAULT_H, 1.0, DEFAULT_N_SIM, DEFAULT_BLOCK, 0.0)
try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        horizon=w.IntSlider(min=8, max=max(DEFAULT_H * 2, 16), value=DEFAULT_H, step=1, description="H"),
        leverage=w.FloatSlider(min=0.25, max=3.0, value=1.0, step=0.25, description="k"),
        n_simulations=w.IntSlider(min=50, max=2000, value=DEFAULT_N_SIM, step=50, description="n_sim"),
        mean_block_length=w.FloatSlider(min=2.0, max=30.0, value=DEFAULT_BLOCK, step=1.0, description="block L"),
        haircut_bps=w.FloatSlider(min=0.0, max=5.0, value=0.0, step=0.25, description="haircut bps"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); default run already executed" % exc)


Headline (no prop-firm pass rates)


,value
hist_mean,0.000462
hist_t_stat,3.317786
hist_p_value,0.000907
hist_ci_excludes_zero,True
hist_bootstrap_p_mean_le_0,0.0
excess_mean,-0.000058
excess_t_stat,-0.175055
excess_p_value,0.861037
excess_ci_excludes_zero,False
horizon_mean_terminal,0.030303


Historical strategy EV significance


,mean,mean_ann,t_stat,p_value,ci_low,ci_high,ci_excludes_zero,bootstrap_p_mean_le_0,bootstrap_ci_low,bootstrap_ci_high,n_obs,psr,psr_vs_1
0,0.000462,0.116498,3.317786,0.000907,0.000189,0.000735,True,0.0,0.000225,0.000711,1164,1.0,0.996904


Excess vs SPY (expectation), separate from P(not beat SPY)


...

EV concentration (mean vs median vs CVaR; top-decile share)


,value
mean_terminal,0.030303
median_terminal,0.026215
cvar_5,-0.032566
top_decile_ev_share,0.341036
top_n,40.000000


Joint shape vs SPY


,value
beta_median,0.016930
corr_median,0.041074
down_capture_median,0.000329
p_not_beat_given_spy_underwater,0.013514
p_spy_terminal_underwater,0.370000
n_paths,400.000000


interactive(children=(IntSlider(value=63, description='H', max=126, min=8), FloatSlider(value=1.0, description…

## 5. Pathwise holes

Not $P(\mathrm{ever\ underwater})$. Max-DD distribution, time spent below the peak, bars to recover from the trough, and the scatter of **terminal wealth vs max DD** (is $E[W_H]$ bought with a deep hole?). Top-decile EV share flags a fragile right tail.


In [5]:
if not PACK:
    raise RuntimeError("run() did not populate PACK")
print("holes head")
display(PACK["holes"].head())
display(PACK["holes_summary"].to_frame("value"))
display(PACK["concentration"].to_frame("value"))
display(PACK["max_dd_hist"])
display(PACK["dd_scatter"])


holes head


,terminal_wealth,terminal_return,max_dd,frac_in_drawdown,frac_below_start,bars_to_recover
simulation,,,,,,
sim_0,1.006566,0.006566,-0.035167,0.936508,0.555556,NaN
sim_1,1.142172,0.142172,-0.013610,0.698413,0.126984,1.0
sim_2,1.016720,0.016720,-0.029421,0.936508,0.190476,3.0
sim_3,1.054741,0.054741,-0.013982,0.746032,0.000000,5.0
sim_4,0.981778,-0.018222,-0.025851,0.920635,0.857143,NaN


,value
max_dd_mean,-0.023089
max_dd_median,-0.021051
max_dd_p10,-0.037757
max_dd_p05,-0.043540
frac_in_dd_median,0.857143
frac_below_start_median,0.174603
bars_to_recover_median,4.000000
bars_to_recover_p90,25.000000
p_never_recover,0.417500
n_paths,400.000000


,value
mean_terminal,0.030303
median_terminal,0.026215
cvar_5,-0.032566
top_decile_ev_share,0.341036
top_n,40.000000


## 6. Joint shape vs SPY

Same paired bootstrap columns as $P(\mathrm{not\ beat\ SPY})$. Median pathwise beta/corr, down-market capture (mean strategy return on bars with SPY $<0$), and $P(W_s\le W_{\mathrm{spy}}\mid W_{\mathrm{spy}}<1)$. Excess-wealth fan is $W_s/W_{\mathrm{spy}}$.


In [6]:
display(PACK["joint_shape"].to_frame("value"))
display(PACK["excess_fan"])
print("OOS terminal percentile among simulated paths", PACK["headline"]["oos_terminal_percentile"])


,value
beta_median,0.016930
corr_median,0.041074
down_capture_median,0.000329
p_not_beat_given_spy_underwater,0.013514
p_spy_terminal_underwater,0.370000
n_paths,400.000000


OOS terminal percentile among simulated paths 72.5


## 7. Evaluation

- Historical $E[r]$ with HAC t/p and `ci_excludes_zero`
- Bootstrap $P^*(\mu\le 0)$
- Horizon $E[W_H-1]$ vs median vs CVaR; top-decile EV share
- Pathwise max DD (median / 5th percentile) and terminal-vs-DD scatter
- Excess-wealth fan and $P(\mathrm{not\ beat\ SPY})$ plus down-market capture
- Sealed OOS overlay on the fan; OOS terminal percentile vs the storm

S2 sealed series is **daily** net book returns, aligned to SPY close-to-close.


In [7]:
# Optional HMM (univariate strategy only — not for P(beat SPY))
from risk.analytics.monte_carlo.hmm_simulator import GaussianHMMSimulator
from risk.analytics.monte_carlo.ev_stats import horizon_ev, cvar, terminal_simple_return

hmm = GaussianHMMSimulator(n_simulations=200, random_seed=0)
hmm.fit(FRAME["strategy"])
hmm_paths = hmm.simulate(DEFAULT_H)
print(hmm.summary(hmm_paths))
print("HMM horizon EV (strategy only)", horizon_ev(hmm_paths))
print("HMM CVaR5", cvar(terminal_simple_return(hmm_paths), alpha=0.05))


          n_paths  mean_wealth  p05_wealth  p25_wealth  p50_wealth  p75_wealth  p95_wealth
asset                                                                                     
strategy      200     1.029991    0.962814    0.998801    1.027254    1.055838    1.109619
HMM horizon EV (strategy only) {'mean_terminal': 0.029991109187526472, 'ci_low': -0.043851187794905266, 'ci_high': 0.12734695020281517, 'n_paths': 200}
HMM CVaR5 -0.04897812148238416
